# ML Lab 9
## Name - Manan Chahal
## Roll No - 23102C0044

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/content/test.csv')
test_df = pd.read_csv('/content/train.csv')

In [ ]:
df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55
3,594197,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),84.10,6457.15
4,594198,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,90.35,1233.65


# Data Preprocessing

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler


print(f"Training columns: {test_df.columns.tolist()}")
print(f"Test columns: {df.columns.tolist()}")

test_df = test_df.drop(columns=['id'], errors='ignore')
df = df.drop(columns=['id'], errors='ignore')


def impute_data(data):

    num_cols = data.select_dtypes(include=['int64', 'float64']).columns
    for col in num_cols:
        data[col] = data[col].fillna(data[col].median())


    cat_cols = data.select_dtypes(include=['object']).columns
    for col in cat_cols:
        data[col] = data[col].fillna(data[col].mode()[0])
    return data

test_df = impute_data(test_df)
df = impute_data(df)

print("Missing values handled and 'id' column dropped.")

Training columns: ['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
Test columns: ['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
Missing values handled and 'id' column dropped.


In [ ]:

y_train = test_df['Churn']
X_train = test_df.drop(columns=['Churn'])
X_test = df.copy()


cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

X_train_encoded = pd.get_dummies(X_train, columns=cat_cols)
X_test_encoded = pd.get_dummies(X_test, columns=cat_cols)


X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print(f'Encoded features shape: {X_train_scaled.shape}')
print('Categorical encoding and normalization completed.')

Encoded features shape: (225115, 46)
Categorical encoding and normalization completed.


In [ ]:
from sklearn.decomposition import PCA


pca = PCA(n_components=0.95, random_state=42)


X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)


n_components = pca.n_components_
cumulative_variance = pca.explained_variance_ratio_.sum()

print(f'Original number of features: {X_train_scaled.shape[1]}')
print(f'Reduced number of components: {n_components}')
print(f'Cumulative explained variance: {cumulative_variance:.4f}')

Original number of features: 46
Reduced number of components: 19
Cumulative explained variance: 0.9638


In [ ]:
from sklearn.model_selection import train_test_split


X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_pca, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f'X_train_split shape: {X_train_split.shape}')
print(f'X_val shape: {X_val.shape}')
print(f'y_train_split shape: {y_train_split.shape}')
print(f'y_val shape: {y_val.shape}')

X_train_split shape: (180092, 19)
X_val shape: (45023, 19)
y_train_split shape: (180092,)
y_val shape: (45023,)


# Outlier Removal with DBSCAN



In [ ]:
from sklearn.cluster import DBSCAN


dbscan = DBSCAN(eps=3.0, min_samples=10, n_jobs=-1)


clusters = dbscan.fit_predict(X_train_split)


outlier_mask = (clusters == -1)
n_outliers = sum(outlier_mask)


X_train_cleaned = X_train_split[~outlier_mask]
y_train_cleaned = y_train_split[~outlier_mask]

print(f'Number of outliers removed: {n_outliers}')
print(f'Shape of X_train_cleaned: {X_train_cleaned.shape}')
print(f'Shape of y_train_cleaned: {y_train_cleaned.shape}')

Number of outliers removed: 1900
Shape of X_train_cleaned: (178192, 19)
Shape of y_train_cleaned: (178192,)


# Model Training


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)


clf.fit(X_train_cleaned, y_train_cleaned)


y_val_pred = clf.predict(X_val)


val_accuracy = accuracy_score(y_val, y_val_pred)
print(f'Validation Accuracy: {val_accuracy:.4f}')

Validation Accuracy: 0.8322


# Prediction

In [ ]:

final_test_df = pd.read_csv('/content/test.csv')
test_ids = final_test_df['id']
X_test_final = final_test_df.drop(columns=['id'], errors='ignore')


num_cols = X_test_final.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols:
    X_test_final[col] = X_test_final[col].fillna(X_test_final[col].median())

cat_cols_local = X_test_final.select_dtypes(include=['object']).columns
for col in cat_cols_local:
    X_test_final[col] = X_test_final[col].fillna(X_test_final[col].mode()[0])


X_test_final_encoded = pd.get_dummies(X_test_final, columns=cat_cols)

_, X_test_final_encoded = X_train_encoded.align(X_test_final_encoded, join='left', axis=1, fill_value=0)

X_test_final_scaled = scaler.transform(X_test_final_encoded)
X_test_final_pca = pca.transform(X_test_final_scaled)


final_predictions = clf.predict(X_test_final_pca)

submission_df = pd.DataFrame({
    'id': test_ids,
    'Churn': final_predictions
})

submission_df.to_csv('submissions.csv', index=False)
print(f'Submissions file generated successfully with {len(submission_df)} rows.')
print(submission_df.head())

Submissions file generated successfully with 254655 rows.
       id Churn
0  594194    No
1  594195    No
2  594196    No
3  594197    No
4  594198    No


In [ ]:
label_map = {'No': 0, 'Yes': 1}
submission_df['Churn'] = submission_df['Churn'].map(label_map)

print('Unique values in Churn column:', submission_df['Churn'].unique())
display(submission_df.head())

submission_df.to_csv('submissions.csv', index=False)
print('Updated submissions.csv saved with numeric labels.')

Unique values in Churn column: [0 1]


,id,Churn
0,594194,0
1,594195,0
2,594196,0
3,594197,0
4,594198,0


Updated submissions.csv saved with numeric labels.
